In [215]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score, precision_score, recall_score, mean_absolute_percentage_error

In [216]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [217]:
file_path = "WineQT.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "yasserh/wine-quality-dataset",
  file_path,
)

print("First 5 records:", df.head())

/tmp/ipykernel_526/2456761086.py:3: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'wine-quality-dataset' dataset.
First 5 records:    fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.5

In [218]:
X = df.drop('quality', axis=1)
y = df["quality"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [219]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

In [220]:
learning_rate = 0.1
epochs = 2000
hidden_layer_size = 64

In [221]:
def neural_network(X_train, y_train):
    num_samples, input_size = X_train.shape
    output_size = 1

    a1 = np.random.randn(input_size, hidden_layer_size) * np.sqrt(2 / input_size)
    b1 = np.zeros((1, hidden_layer_size))

    a2 = np.random.randn(hidden_layer_size, output_size) * np.sqrt(2 / hidden_layer_size)
    b2 = np.zeros((1, output_size))

    y_train_reshaped = y_train.to_numpy().reshape(-1, 1)

    for epoch in range(epochs):
        hidden = np.dot(X_train, a1) + b1
        A1 = np.maximum(0, hidden)

        output = np.dot(A1, a2) + b2
        y_pred = output

        error_gradient = (y_pred - y_train_reshaped) / num_samples

        d_output = error_gradient

        d_a2 = np.dot(A1.T, d_output)
        d_b2 = np.sum(d_output, axis=0).reshape(1, -1)

        d_A1 = np.dot(d_output, a2.T)
        d_hidden = d_A1 * (hidden > 0).astype(float)

        d_a1 = np.dot(X_train.T, d_hidden)
        d_b1 = np.sum(d_hidden, axis=0).reshape(1, -1)

        a1 -= learning_rate * d_a1
        b1 -= learning_rate * d_b1
        a2 -= learning_rate * d_a2
        b2 -= learning_rate * d_b2

    return a1, b1, a2, b2

def predict_nn(X, a1, b1, a2, b2):
    hidden = np.dot(X, a1) + b1
    A1 = np.maximum(0, hidden)

    y_pred = np.dot(A1, a2) + b2
    return y_pred.flatten()

In [222]:
a1, b1, a2, b2 = neural_network(X_train, y_train)
y_pred_train = predict_nn(X_train, a1, b1, a2, b2)

In [223]:
mse = mean_squared_error(y_train, y_pred_train)
r2 = r2_score(y_train, y_pred_train)
mape = mean_absolute_percentage_error(y_train, y_pred_train)

In [224]:
print(f"MSE: {mse:.4f}")
print(f"R2_score: {r2:.4f}")
print("MAPE:", round(mape, 3))

MSE: 0.2053
R2_score: 0.6840
MAPE: 0.064
